# 참조 얼굴 4차 — 메이크업·표정 축

3차(candidates_0825, A1 ~ F2) 는 얼굴형 6종 × 2장을 no makeup·무표정으로 뽑았고,
팀 피드백은 "B2·D2·E2 예쁨, 12장 비슷함, 메이크업 있으면 더 예쁨, 웃는 참조 섞기".
4차는 얼굴형 B·D·E 를 고정하고 메이크업 3종 × 표정으로 9장 + 쌍꺼풀 명시 2장을 뽑아
B2·D2·E2 와 합쳐 12장(ref-39~50)을 만든다.

In [ ]:
import base64
import io
import time
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
from google.colab import drive
from openai import OpenAI
from PIL import Image

drive.mount("/content/drive")
REF_DIR = Path("/content/drive/MyDrive/saloncut_data/ref_faces")
CAND4 = REF_DIR / "candidates_0826"
CAND4.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=getpass("OpenAI API key: "))

BEAUTY3 = (
    "a Korean woman in her early 20s, breathtakingly beautiful, "
    "K-pop idol visual center level beauty, the kind of face that stops people "
    "in the street, perfect facial harmony and golden-ratio proportions, "
    "flawless luminous skin, captivating eyes"
)

FACES3 = {
    "B": "defined V-line face with gentle cheekbones, deep-set expressive eyes with long lashes, sculpted nose, full lips, glamorous impression",
    "D": "slim face with refined bone structure, elegant upturned almond eyes, thin high nose bridge, defined lips, sophisticated cat-like charm",
    "E": "small heart-shaped face, big round sparkling eyes, petite straight nose, small rosy lips, doll-like lovely beauty",
}

SUFFIX4_BASE = (
    "completely bald with no hair on the head, eyebrows present, "
    "facing directly forward, eyes open looking at camera, "
    "{expr}, head and shoulders only, plain black top, "
    "solid light gray background, even soft studio lighting, minimal shadows, "
    "{makeup}, no accessories, no glasses, photorealistic portrait photograph, "
    "sharp focus on the face"
)

MAKEUP4 = {
    "nat": ("natural everyday makeup, soft brows, tinted lips, clean dewy skin",
            "neutral relaxed expression"),
    "idol": ("K-pop idol stage makeup, defined eyeliner, long lashes, glossy coral lips, luminous highlighted skin",
             "neutral relaxed expression"),
    "smile": ("K-pop idol stage makeup, defined eyeliner, long lashes, glossy coral lips, luminous highlighted skin",
              "gentle natural closed-lip smile, warm expression"),
}


def build_prompt(face, mk, eyelid=None):
    makeup, expr = MAKEUP4[mk]
    detail = FACES3[face] if eyelid is None else f"{FACES3[face]}, {eyelid}"
    return f"{BEAUTY3}, {detail}, {SUFFIX4_BASE.format(expr=expr, makeup=makeup)}"


def gen(key, prompt):
    t = time.time()
    res = client.images.generate(model="gpt-image-2", prompt=prompt, size="1024x1024", n=1)
    img = Image.open(io.BytesIO(base64.b64decode(res.data[0].b64_json))).convert("RGB")
    img.save(CAND4 / f"{key}.png")
    print(f"{key}  {time.time() - t:.0f}s")
    return img


print(build_prompt("B", "idol"))

In [ ]:
cands4 = {}
for face in ["B", "D", "E"]:
    for mk in MAKEUP4:
        key = f"{face}_{mk}"
        cands4[key] = gen(key, build_prompt(face, mk))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, (k, img) in zip(axes.flat, cands4.items()):
    ax.imshow(img); ax.set_title(k, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(CAND4 / "grid_candidates_0826.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
EXTRA4 = {
    "B_idol_dbl": ("B", "idol", "clearly visible double eyelids"),
    "E_nat_dbl": ("E", "nat", "wide parallel double eyelids"),
}
for key, (face, mk, eyelid) in EXTRA4.items():
    cands4[key] = gen(key, build_prompt(face, mk, eyelid))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, key in zip(axes, EXTRA4):
    ax.imshow(cands4[key]); ax.set_title(key, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(CAND4 / "grid_extra_0826.png", dpi=100, bbox_inches="tight")
plt.show()

# 4차 참조 후보 × 매장 모델 5장 조합 3 검증

candidates_0826 11장을 참조로 salon 01~05 에 조합 3(st 0.4 / ip 0.5 / cn 0.8)을 돌린다.
dev 기준 `_run_reference_mode` 와 같은 흐름(2048 후처리·복원 포함). 시드는 서비스와 같이 랜덤.

In [ ]:
%cd /content
import importlib
import os
import random
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
from google.colab import drive
from PIL import Image

drive.mount("/content/drive")

REPO = "/content/SalonCutAI"
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q -b dev https://github.com/qja0707/SalonCutAI.git {REPO}
!pip install -q diffusers==0.39.0 transformers==5.14.1 peft==0.19.1 accelerate==1.14.0 \
    insightface onnxruntime mediapipe==1.0.0 opencv-contrib-python-headless \
    facexlib torchvision

os.environ["IMAGE_GEN_ENABLED"] = "1"
os.environ["SALON_STORAGE_DIR"] = "/content/storage"
sys.path.insert(0, f"{REPO}/backend")
importlib.invalidate_caches()

from src.ai_engine.image_gen import combo3, downloads, loader, masks, pipeline, storage

downloads.ensure_models()
print("missing:", downloads.missing_files())
loader.get_face_app()
loader.get_landmarker()
loader.get_segmenter()
loader.get_codeformer()
loader.get_face_helper()
loader.get_combo3()

import torch
print(f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
SALON = Path("/content/drive/MyDrive/saloncut_data/test_images/salon")
CAND4 = Path("/content/drive/MyDrive/saloncut_data/ref_faces/candidates_0826")
OUT = Path("/content/drive/MyDrive/saloncut_data/outputs/refcheck_0826")
OUT.mkdir(parents=True, exist_ok=True)

TARGETS = ["salon_01_long_wave_brown", "salon_02_long_wave_dark", "salon_03_long_wave_ring",
           "salon_04_long_wave_black", "salon_05_short_bob_brown"]
CANDS = ["B_nat", "B_idol", "B_smile", "D_nat", "D_idol", "D_smile",
         "E_nat", "E_idol", "E_smile", "B_idol_dbl", "E_nat_dbl"]


def run_with_ref(src, ref_path, seed):
    """dev 의 _run_reference_mode 와 같은 흐름. 참조 경로만 직접 받는다."""
    out, _, _ = combo3.generate(src, ref_path, seed)
    face_mask = masks.build_face_mask(src)
    hair_mask = masks.build_hair_mask(src)
    return pipeline._postprocess(src, out, face_mask, hair_mask)


srcs = {n: storage.to_stored_size(Image.open(SALON / f"{n}.jpg").convert("RGB")) for n in TARGETS}
seeds = {}

for name in TARGETS:
    src = srcs[name]
    fig, axes = plt.subplots(2, len(CANDS) + 1, figsize=(3.2 * (len(CANDS) + 1), 9))
    axes[0, 0].imshow(src); axes[0, 0].set_title(f"{name[:8]} src", fontsize=10)
    axes[1, 0].axis("off")
    for col, cand in enumerate(CANDS, start=1):
        ref_path = CAND4 / f"{cand}.png"
        seed = random.randint(0, 2**31 - 1)
        seeds[(name, cand)] = seed
        t = time.time()
        fin = run_with_ref(src, ref_path, seed)
        fin.save(OUT / f"{name}_{cand}.png")
        axes[0, col].imshow(Image.open(ref_path)); axes[0, col].set_title(cand, fontsize=10)
        axes[1, col].imshow(fin); axes[1, col].set_title(f"seed {seed}", fontsize=8)
        print(f"{name[:8]}  {cand}  {time.time() - t:.0f}s")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / f"grid_{name}_refcheck.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
FACES3.update({
    "G": "youthful clean-cut face, large bright eyes with defined parallel double eyelids and clear whites, straight slim nose, small full lips with natural pink tint, smooth clear skin, fresh innocent impression",
    "H": "small round face with soft cheeks, big round doe eyes with visible aegyo-sal under the eyes, subtle rosy blush on the cheeks, small pouty lips, cute puppy-like impression",
})
MAKEUP4["glam"] = (
    "soft glam makeup, aegyo-sal highlight under the eyes, peachy blush, glossy pink lips, luminous skin",
    "neutral relaxed expression",
)

PLAN = [("G", "nat"), ("G", "idol"), ("G", "smile"), ("H", "nat"), ("H", "glam"), ("H", "smile")]
for face, mk in PLAN:
    key = f"{face}_{mk}"
    cands4[key] = gen(key, build_prompt(face, mk))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (face, mk) in zip(axes.flat, PLAN):
    key = f"{face}_{mk}"
    ax.imshow(cands4[key]); ax.set_title(key, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(CAND4 / "grid_GH_0826.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
FACES3.update({
    "G": "youthful clean-cut face, large bright eyes with defined parallel double eyelids and clear whites, straight slim nose, small full lips with natural pink tint, smooth clear skin, fresh innocent impression",
    "H": "small round face with soft cheeks, big round doe eyes with visible aegyo-sal under the eyes, subtle rosy blush on the cheeks, small pouty lips, cute puppy-like impression",
})
MAKEUP4["glam"] = (
    "soft glam makeup, aegyo-sal highlight under the eyes, peachy blush, glossy pink lips, luminous skin",
    "neutral relaxed expression",
)

cands4 = {"G_nat": Image.open(CAND4 / "G_nat.png")}
PLAN = [("G", "idol"), ("G", "smile"), ("H", "nat"), ("H", "glam"), ("H", "smile")]
for face, mk in PLAN:
    key = f"{face}_{mk}"
    cands4[key] = gen(key, build_prompt(face, mk))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, key in zip(axes.flat, cands4):
    ax.imshow(cands4[key]); ax.set_title(key, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(CAND4 / "grid_GH_0826.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
CANDS_GH = ["G_nat", "G_idol", "G_smile", "H_nat", "H_glam", "H_smile"]

for name in TARGETS:
    src = srcs[name]
    fig, axes = plt.subplots(2, len(CANDS_GH) + 1, figsize=(3.2 * (len(CANDS_GH) + 1), 9))
    axes[0, 0].imshow(src); axes[0, 0].set_title(f"{name[:8]} src", fontsize=10)
    axes[1, 0].axis("off")
    for col, cand in enumerate(CANDS_GH, start=1):
        ref_path = CAND4 / f"{cand}.png"
        seed = random.randint(0, 2**31 - 1)
        seeds[(name, cand)] = seed
        t = time.time()
        fin = run_with_ref(src, ref_path, seed)
        fin.save(OUT / f"{name}_{cand}.png")
        axes[0, col].imshow(Image.open(ref_path)); axes[0, col].set_title(cand, fontsize=10)
        axes[1, col].imshow(fin); axes[1, col].set_title(f"seed {seed}", fontsize=8)
        print(f"{name[:8]}  {cand}  {time.time() - t:.0f}s")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT / f"grid_{name}_refcheck_GH.png", dpi=100, bbox_inches="tight")
    plt.show()